# Dokumentenverarbeitungs-Pipeline — Siemens-Handbuch RAG

Dieses Notebook leitet die **Offline-Aufbereitung** des lokalen Siemens-Troubleshooting-
Assistenten Schritt für Schritt ab und ist eigenständig in **Google Colab** lauffähig
(kein LM Studio nötig — es zeigt die Dokumentenverarbeitung bis zum durchsuchbaren,
re-rankten Retrieval).

**Pipeline:** PDF → Docling → Markdown → Chunking (inkl. Tabellen-Explosion) →
lokale Embeddings (e5) → Vektorindex → Hybrid-Retrieval (Vektor + Fehlercode-Lookup)
→ Cross-Encoder-Reranking → Guardrail → (optional) Antwortgenerierung.

Entspricht `parser.py` + `rag_engine.py` des Repos.

> Tipp: **Laufzeit → Laufzeittyp ändern → GPU** beschleunigt Docling, Embeddings und
> Reranker deutlich.

## Setup

Installiert Docling (PDF-Extraktion), LlamaIndex (Chunking/Index/Retrieval) und
sentence-transformers (Embeddings + Reranker). Der erste Lauf lädt einmalig Modelle.

In [ ]:
!pip -q install docling "llama-index-core>=0.11" "llama-index-embeddings-huggingface>=0.3" "sentence-transformers>=3.0"
print("✅ Installation fertig")

## Schritt 1 — PDF bereitstellen

Lade `siemens-handbuch.pdf` hoch (Colab-Upload-Dialog) **oder** setze weiter unten
eine `PDF_URL`.

In [ ]:
import os
PDF_PATH = "siemens-handbuch.pdf"

# Option A: in Colab hochladen
try:
    from google.colab import files
    if not os.path.exists(PDF_PATH):
        print("Bitte siemens-handbuch.pdf hochladen …")
        uploaded = files.upload()
        PDF_PATH = next(iter(uploaded))
except ImportError:
    pass

# Option B: alternativ von einer URL laden (auskommentieren und URL setzen)
# import urllib.request
# PDF_URL = "https://example.org/siemens-handbuch.pdf"
# urllib.request.urlretrieve(PDF_URL, PDF_PATH)

assert os.path.exists(PDF_PATH), "Kein PDF gefunden — hochladen oder PDF_URL setzen."
print("PDF:", PDF_PATH, "|", os.path.getsize(PDF_PATH) // 1024, "KB")

## Schritt 2 — Extraktion mit Docling (`parser.py`)

Das Handbuch ist visuell gesetzt (mehrspaltig, Tabellen, Symbole). Docling analysiert
das Layout KI-gestützt (Lesereihenfolge, Tabellen, OCR) und liefert **strukturiertes
Markdown** — verlustarm, menschlich prüfbar und strukturbewusst chunkbar.

In [ ]:
import time
from docling.document_converter import DocumentConverter

t0 = time.time()
converter = DocumentConverter()
result = converter.convert(PDF_PATH)          # lädt beim ersten Mal Layout-Modelle
markdown = result.document.export_to_markdown()

with open("siemens_wissen.md", "w", encoding="utf-8") as f:
    f.write(markdown)

print(f"✅ Docling fertig in {time.time()-t0:.1f}s — {len(markdown):,} Zeichen")
print("--- Auszug ---")
print(markdown[:600])

## Schritt 3 — Struktur der Wissensbasis verstehen

Docling erzeugt `##`-Abschnitte und Markdown-Tabellen. Die **Fehlercodes und
Störungen stehen in großen Tabellen** — das ist gleich fürs Chunking entscheidend.

In [ ]:
lines = markdown.splitlines()
h2  = [l for l in lines if l.startswith("## ")]
tbl = [l for l in lines if l.strip().startswith("|")]
print(f"## Abschnitte: {len(h2)} | Tabellenzeilen: {len(tbl)}")
print("Beispiel-Überschriften:", [h[3:50] for h in h2[:6]])

idx = next((i for i, l in enumerate(lines) if "E:18" in l), None)
if idx is not None:
    print("\nFehlercode-Zeile (roh, gekürzt):\n", lines[idx][:180])

## Schritt 4 — Chunking: Markdown-Struktur + Tabellen-Explosion

Zwei Stufen:

1. **`MarkdownNodeParser`** schneidet an den `##`-Überschriften.
2. **Tabellen-Explosion (entscheidend):** Die Fehlercode-Tabelle landet sonst als *ein*
   ~6.400-Zeichen-Block und embeddet als „Brei" — eine Frage wie „Fehler E:18" findet
   ihn dann nicht. Deshalb werden große tabellenlastige Nodes **pro Zeile** aufgeteilt
   und als lesbarer Fließtext gerendert (statt gepaddter Rohzeile).

In [ ]:
import re
from llama_index.core import Document
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.core.schema import TextNode

MAX_NODE_CHARS = 1600

def split_row(line):
    return [re.sub(r"\s+", " ", c).strip() for c in line.strip().strip("|").split("|")]

def explode_markdown_tables(nodes, max_chars=MAX_NODE_CHARS):
    out = []
    for node in nodes:
        text = node.get_content()
        table_lines = [l for l in text.split("\n") if l.strip().startswith("|")]
        if len(text) <= max_chars or len(table_lines) < 3:
            out.append(node)
            continue
        parts = text.split("\n")
        heading = " ".join(l.strip() for l in parts if l.strip().startswith("#"))
        prose = "\n".join(l for l in parts
                          if l.strip() and not l.strip().startswith("|") and not l.strip().startswith("#")).strip()
        labels = split_row(table_lines[0])
        rows = table_lines[2:]                       # [1] ist die |---|-Trennzeile
        if prose:
            out.append(TextNode(text=f"{heading}\n{prose}".strip()))
        for row in rows:
            cells = split_row(row)
            if not any(cells):
                continue
            pairs = "; ".join(f"{lbl}: {val}" if lbl else val
                              for lbl, val in zip(labels + [""] * len(cells), cells) if val)
            out.append(TextNode(text=f"{heading}\n{pairs}".strip()))
    return out

base_nodes = MarkdownNodeParser().get_nodes_from_documents([Document(text=markdown)])
nodes = explode_markdown_tables(base_nodes)
print(f"Nodes: {len(base_nodes)} → {len(nodes)} (nach Tabellen-Explosion)")

e18 = [n for n in nodes if "E:18" in n.get_content()]
print("\nBeispiel — E:18 ist jetzt ein eigener Chunk:\n", (e18[0].get_content()[:300] if e18 else "—"))

## Schritt 5 — Lokale Embeddings mit e5-Präfixen

Jeder Node wird lokal vektorisiert mit `intfloat/multilingual-e5-small`.

**Wichtig:** e5 ist asymmetrisch trainiert und verlangt Präfixe — Fragen `query: …`,
Textstücke `passage: …`. Ohne diese Präfixe sinkt die Trefferqualität deutlich.

In [ ]:
from llama_index.core import Settings, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(
    model_name="intfloat/multilingual-e5-small",
    query_instruction="query: ",
    text_instruction="passage: ",
)

index = VectorStoreIndex(nodes)          # embeddet alle Nodes lokal
print("✅ Vektorindex gebaut:", len(nodes), "Nodes")

## Schritt 6 — Persistenz (optional)

In der App wird der Index nach `storage/` persistiert und nur bei geänderter Quelle
oder geändertem Embedding-Modell neu gebaut (Hash-Invalidierung).

In [ ]:
index.storage_context.persist("storage")
print("Persistiert nach ./storage — Neuladen via "
      "load_index_from_storage(StorageContext.from_defaults(persist_dir='storage'))")

## Schritt 7 — Hybrides Retrieval (Vektor + Fehlercode-Lookup)

Dense-Embeddings sind bei seltenen Tokens wie „E:18" schwach. Enthält die Frage einen
Fehlercode, wird der Chunk mit genau diesem Code **garantiert** ins Kandidatenset
gelegt — sonst würde er nicht gefunden.

In [ ]:
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.schema import NodeWithScore

CODE_RE = re.compile(r"(?:\bE[:\s]?|\bfehler(?:code)?\s+)(\d{1,3})\b", re.IGNORECASE)

def extract_error_codes(q):
    codes = set()
    for m in CODE_RE.finditer(q or ""):
        codes.add(f"E:{m.group(1)}")
        codes.add(f"E{m.group(1)}")
    return codes

class HybridRetriever(BaseRetriever):
    def __init__(self, index, k=12):
        self._vr = index.as_retriever(similarity_top_k=k)
        self._docs = index.docstore.docs
        super().__init__()
    def _retrieve(self, qb):
        found = self._vr.retrieve(qb)
        codes = extract_error_codes(qb.query_str)
        if codes:
            have = {n.node.node_id for n in found}
            for nid, nd in self._docs.items():
                if nid not in have and any(c in nd.get_content() for c in codes):
                    found.append(NodeWithScore(node=nd, score=1.0))
        return found

retriever = HybridRetriever(index, k=12)
res = retriever.retrieve("Was bedeutet der Fehlercode E:18?")
print("Kandidaten:", len(res), "| E:18 im Set?:",
      any("E:18" in r.node.get_content() for r in res))

## Schritt 8 — Cross-Encoder-Reranking

Ein multilingualer Reranker (`BAAI/bge-reranker-v2-m3`) liest **Frage und Chunk
gemeinsam** und ordnet die Kandidaten weit präziser als reine Vektorähnlichkeit.
Er reduziert auf die relevantesten `top_n` Chunks — genau das, was das LLM sieht.

> Erster Lauf lädt den Reranker (~2 GB).

In [ ]:
from llama_index.core.postprocessor import SentenceTransformerRerank

reranker = SentenceTransformerRerank(model="BAAI/bge-reranker-v2-m3", top_n=5)

q = "Was bedeutet der Fehlercode E:18?"
top5 = reranker.postprocess_nodes(retriever.retrieve(q), query_str=q)
print("Top-Chunk nach Rerank (Score", round(top5[0].score, 3), "):\n")
print(top5[0].node.get_content()[:280])

## Schritt 9 — Guardrail gegen Halluzination

Der Reranker liefert Sigmoid-Scores in `[0,1]`. Kalibriert: In-Scope-Fragen scoren
hoch, fachfremde ~0.0. Unter der Schwelle `GUARDRAIL_MIN_SCORE` wird „nicht im
Handbuch" geantwortet — ohne LLM-Aufruf.

In [ ]:
GUARDRAIL_MIN_SCORE = 0.15

def top_score(q):
    r = reranker.postprocess_nodes(retriever.retrieve(q), query_str=q)
    return r[0].score if r else 0.0

for q in ["Fehler E:23", "Wie reinige ich die Trommel?",
          "Wie backe ich einen Kuchen?", "Hauptstadt von Australien?"]:
    s = top_score(q)
    tag = "✅ im Handbuch" if s >= GUARDRAIL_MIN_SCORE else "⛔ Guardrail: nicht im Handbuch"
    print(f"{s:+.3f}  {tag:32s} | {q}")

## Schritt 10 — (Optional) Antwortgenerierung

In der echten App generiert ein **lokales Modell in LM Studio** die Antwort aus dem
Kontext. In Colab lässt sich das optional mit einem kleinen HF-Modell demonstrieren
(GPU-Runtime empfohlen). Standardmäßig wird nur der assemblierte Kontext gezeigt.

In [ ]:
GENERATE = False   # auf True setzen für eine echte Antwort (lädt ein kleines Modell)

q = "Was bedeutet der Fehlercode E:18 und was soll ich tun?"
ctx = "\n\n".join(n.node.get_content()
                  for n in reranker.postprocess_nodes(retriever.retrieve(q), query_str=q))

if GENERATE:
    !pip -q install "transformers>=4.44" accelerate
    import torch
    from transformers import pipeline
    gen = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct",
                   torch_dtype=torch.float16, device_map="auto")
    prompt = (f"Nutze NUR den folgenden Handbuch-Kontext. Antworte auf Deutsch in "
              f"Stichpunkten.\n\nKontext:\n{ctx}\n\nFrage: {q}\nAntwort:")
    text = gen(prompt, max_new_tokens=300, do_sample=False)[0]["generated_text"]
    print(text[len(prompt):])
else:
    print("GENERATE=False — nur Kontext-Assembly (Generierung übersprungen).\n")
    print("Assemblierter Kontext (erste 500 Zeichen):\n")
    print(ctx[:500])

## Zusammenfassung

Diese Pipeline entspricht dem Repo:

| Notebook-Schritt | Repo |
|------------------|------|
| 2 Docling-Extraktion | `parser.py` |
| 4 Chunking + Tabellen-Explosion | `rag_engine._explode_markdown_tables` |
| 5 Embeddings (e5-Präfixe) | `rag_engine.get_embed_model` |
| 5–6 Index + Persistenz | `rag_engine.build_or_load_index` |
| 7 Hybrid-Retrieval | `rag_engine.make_retriever` |
| 8 Reranking | `rag_engine.get_reranker` |
| 9 Guardrail | `rag_engine.is_grounded` |

Für die interaktive Web-App/CLI mit lokalem LLM siehe `server.py`, `lokale_ki.py`
und `docs/TECHNICAL.md`.